##Exercícios
1) Treinamento com GPU

As GPU's (*Graphics processing unit*) possuem características avançadas como renderização de games 3D. As GPU's são otimizadas para processar aplicações e realizar a mesma operação várias vezes, de forma bastante rápida, ideal para processamento iterativo, a base de algoritmos de Machine Learning.

Alterando o ambiente de execução para GPU: vá em **Ambiente de execução** $→$ **Alterar o tipo de ambiente de execução** $→$  **Acelerador de hardware** $→$  **T4 GPU**.

Treine novamente a rede e compare o tempo de treinamento em GPU com o treinamento em CPU (none).

2) Pode-se realizar mudanças nos hiperparâmetros com o intuito de melhorar os resultados obtidos. Ao observar os gráficos gerados ao final do treinamento, pode-se observar que durante as 20 épocas, o modelo não alcançou a convergência, que seria representada por uma linha quase horizontal no final do gráfico. Pode-se aumentrar o número de épocas para 30, 50, 100 ou maior para se tentar melhor acurária. Deve-se tomar cuidado com o overfitting, característico de um treinamento excessivo.

3) Crie figuras de dígitos (0 à 9) e carrege no banco de dados de testes. Faça as predições e veirifique se a rede consegue predizer positivamente seus dígitos.

4) Treine a rede sem efetuar a normalização (reescalonamento) e compare os resultados obtidos.

5) Utilizando os conhecimentos e as experiências adiquiridos nos exercícios com redes neurais anteriores, proceda a sintonia da rede, objetivando uma melhor eficiência. Mantendo a arquitetura da rede, processa a sintonia dos hiperparâmetros: `learning_rate` e `batch_size `tentando uma melhor desempenho da rede. Tome cuidado com o *overfitting*! Elabore gráficos e/ou tabelas comparativas.

6) Modifique as funções de ativação das camadas ocultas pelas funções: *relu*, *linear* e *sigmoid*. Elabore gráficos e/ou tabelas comparativas.

7) Com os melhores resultados conseguidos, proceda a mudança do otimizador substituindo o **SGD** (*Stochastic Gradient Descent*) pelo **Adam**, **Adagrad** e **RMSprop**.  Elabore gráficos e/ou tabelas comparativas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, Adam, Adagrad, RMSprop
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Carregando o dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
print("Classes:", np.unique(y_train))
print("Primeiro rótulo:", y_train[0])

In [ ]:
plt.imshow(X_train[0], cmap='gray')
plt.title(f"Dígito: {y_train[0]}")
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x=y_train)
plt.title("Distribuição das classes no conjunto de treinamento")
plt.xlabel("Dígito")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
X_train_flat = X_train.reshape(60000, 784)
X_test_flat = X_test.reshape(10000, 784)

print("X_train_flat:", X_train_flat.shape)
print("X_test_flat:", X_test_flat.shape)

In [ ]:
print("Mínimo antes:", X_train_flat.min())
print("Máximo antes:", X_train_flat.max())

X_train_sc = X_train_flat / 255.0
X_test_sc = X_test_flat / 255.0

print("Mínimo depois:", X_train_sc.min())
print("Máximo depois:", X_train_sc.max())

In [ ]:
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

print("Rótulo original:", y_train[0])
print("Rótulo convertido:", y_train_cat[0])

In [ ]:
model = Sequential()

# Camada de entrada
model.add(Input(shape=(784,)))

# Primeira camada oculta
model.add(Dense(256, activation='tanh'))

# Segunda camada oculta
model.add(Dense(128, activation='tanh'))

# Camada de saída
model.add(Dense(10, activation='softmax'))

In [ ]:
model.summary()

In [ ]:
optimizer = SGD(learning_rate=0.2)

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

In [ ]:
start_time = time.process_time()

history = model.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    verbose=1
)

training_time = time.process_time() - start_time

print(f"\nTempo de treinamento: {training_time:.2f} segundos")
print(f"Tempo de treinamento: {training_time / 60:.2f} minutos")

In [ ]:
plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Função de custo')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
loss, accuracy = model.evaluate(
    X_test_sc,
    y_test_cat,
    verbose=0
)

print(f"Acurácia no teste: {accuracy * 100:.2f}%")
print(f"Loss no teste: {loss:.4f}")

In [ ]:
prediction = model.predict(X_test_sc)

y_predict = np.argmax(prediction, axis=1)

print("Formato das predições:", prediction.shape)

In [ ]:
amostra = 66

print("Probabilidades:")
print(prediction[amostra])

print("\nClasse prevista:", prediction[amostra].argmax())
print("Probabilidade:", prediction[amostra].max() * 100)
print("Classe real:", y_test[amostra])

In [ ]:
plt.figure(figsize=(10, 8))

for i in range(20):
    plt.subplot(5, 4, i + 1)

    plt.imshow(X_test[i], cmap='gray')

    plt.title(
        f"Real: {y_test[i]} | Previsto: {y_predict[i]}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
confusion = confusion_matrix(y_test, y_predict)

print(confusion)

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    confusion,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Matriz de Confusão")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.show()

In [ ]:
print(classification_report(y_test, y_predict))

##Exercício 1.

In [ ]:
start_time = time.time()

history_cpu = model.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    verbose=1
)

tempo_cpu = time.time() - start_time

print(f"Tempo CPU: {tempo_cpu:.2f} segundos")

In [ ]:
start_time = time.time()

history_gpu = model.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    verbose=1
)

tempo_gpu = time.time() - start_time

print(f"Tempo GPU: {tempo_gpu:.2f} segundos")

In [ ]:
print(f"Tempo CPU: {tempo_cpu:.2f} segundos")
print(f"Tempo GPU: {tempo_gpu:.2f} segundos")

##Exercício 2.

In [ ]:
model_50 = Sequential([
    Input(shape=(784,)),
    Dense(256, activation='tanh'),
    Dense(128, activation='tanh'),
    Dense(10, activation='softmax')
])

model_50.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.2),
    metrics=['accuracy']
)

history_50 = model_50.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history_50.history['accuracy'], label='Treino')
plt.plot(history_50.history['val_accuracy'], label='Validação')

plt.title("Acurácia - 50 épocas")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()
plt.grid()

plt.show()

##Exercício 3.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

img = Image.open('0.png')

# Redimensionar para 28x28
img = img.resize((28, 28))

# Converter para escala de cinza
img = img.convert('L')

# Converter para array
img = np.array(img)

# Inverter os pixels
img = 255 - img

# Visualizar
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
img_flat = img.reshape(1, 784)
img_sc = img_flat / 255.0

prediction_new = model.predict(img_sc)

classe_prevista = np.argmax(prediction_new)

print("Classe prevista:", classe_prevista)
print("Probabilidade:", prediction_new.max() * 100)

##Exercício 4.

In [ ]:
model_sem_normalizacao = Sequential([
    Input(shape=(784,)),
    Dense(256, activation='tanh'),
    Dense(128, activation='tanh'),
    Dense(10, activation='softmax')
])

model_sem_normalizacao.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.2),
    metrics=['accuracy']
)

history_sem_normalizacao = model_sem_normalizacao.fit(
    X_train_flat,
    y_train_cat,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    verbose=1
)

In [ ]:
loss_sem_norm, acc_sem_norm = model_sem_normalizacao.evaluate(
    X_test_flat,
    y_test_cat,
    verbose=0
)

print(f"Acurácia sem normalização: {acc_sem_norm * 100:.2f}%")

In [ ]:
loss_norm, acc_norm = model.evaluate(
    X_test_sc,
    y_test_cat,
    verbose=0
)

print(f"Acurácia com normalização: {acc_norm * 100:.2f}%")
print(f"Acurácia sem normalização: {acc_sem_norm * 100:.2f}%")

##Exercício 5.

In [ ]:
resultados = []

learning_rates = [0.01, 0.05, 0.1, 0.2]
batch_sizes = [64, 128, 256]

for lr in learning_rates:

    for batch in batch_sizes:

        modelo = Sequential([
            Input(shape=(784,)),
            Dense(256, activation='tanh'),
            Dense(128, activation='tanh'),
            Dense(10, activation='softmax')
        ])

        modelo.compile(
            loss='categorical_crossentropy',
            optimizer=SGD(learning_rate=lr),
            metrics=['accuracy']
        )

        inicio = time.time()

        hist = modelo.fit(
            X_train_sc,
            y_train_cat,
            batch_size=batch,
            epochs=10,
            validation_split=0.1,
            verbose=0
        )

        tempo = time.time() - inicio

        resultados.append([
            lr,
            batch,
            hist.history['val_accuracy'][-1],
            tempo
        ])

In [ ]:
import pandas as pd

resultados_df = pd.DataFrame(
    resultados,
    columns=[
        'Learning Rate',
        'Batch Size',
        'Val Accuracy',
        'Tempo (s)'
    ]
)

resultados_df

#Exercício 6.

In [ ]:
ativacoes = ['tanh', 'relu', 'linear', 'sigmoid']

resultados_ativacao = []

for ativacao in ativacoes:

    modelo = Sequential([
        Input(shape=(784,)),
        Dense(256, activation=ativacao),
        Dense(128, activation=ativacao),
        Dense(10, activation='softmax')
    ])

    modelo.compile(
        loss='categorical_crossentropy',
        optimizer=SGD(learning_rate=0.2),
        metrics=['accuracy']
    )

    hist = modelo.fit(
        X_train_sc,
        y_train_cat,
        batch_size=128,
        epochs=10,
        validation_split=0.1,
        verbose=0
    )

    resultados_ativacao.append([
        ativacao,
        hist.history['accuracy'][-1],
        hist.history['val_accuracy'][-1]
    ])

In [ ]:
ativacao_df = pd.DataFrame(
    resultados_ativacao,
    columns=[
        'Ativação',
        'Acurácia Treino',
        'Acurácia Validação'
    ]
)

ativacao_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    ativacao_df['Ativação'],
    ativacao_df['Acurácia Validação']
)

plt.title("Comparação das funções de ativação")
plt.xlabel("Função de ativação")
plt.ylabel("Acurácia de validação")

plt.show()

#Exercício 7.

In [ ]:
otimizadores = {
    'SGD': SGD(learning_rate=0.2),
    'Adam': Adam(),
    'Adagrad': Adagrad(),
    'RMSprop': RMSprop()
}

resultados_otimizadores = []

for nome, otimizador in otimizadores.items():

    modelo = Sequential([
        Input(shape=(784,)),
        Dense(256, activation='tanh'),
        Dense(128, activation='tanh'),
        Dense(10, activation='softmax')
    ])

    modelo.compile(
        loss='categorical_crossentropy',
        optimizer=otimizador,
        metrics=['accuracy']
    )

    inicio = time.time()

    hist = modelo.fit(
        X_train_sc,
        y_train_cat,
        batch_size=128,
        epochs=10,
        validation_split=0.1,
        verbose=0
    )

    tempo = time.time() - inicio

    resultados_otimizadores.append([
        nome,
        hist.history['accuracy'][-1],
        hist.history['val_accuracy'][-1],
        tempo
    ])

In [ ]:
otimizadores_df = pd.DataFrame(
    resultados_otimizadores,
    columns=[
        'Otimizador',
        'Acurácia Treino',
        'Acurácia Validação',
        'Tempo (s)'
    ]
)

otimizadores_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    otimizadores_df['Otimizador'],
    otimizadores_df['Acurácia Validação']
)

plt.title("Comparação dos otimizadores")
plt.xlabel("Otimizador")
plt.ylabel("Acurácia de validação")

plt.show()

In [ ]:
print("===== RESULTADO FINAL =====")

print(f"Acurácia modelo inicial: {acc_norm * 100:.2f}%")
print(f"Acurácia sem normalização: {acc_sem_norm * 100:.2f}%")
print(f"Tempo de treinamento inicial: {training_time:.2f} segundos")

In [ ]:
print("\n===== FUNÇÕES DE ATIVAÇÃO =====")
print(ativacao_df)

print("\n===== OTIMIZADORES =====")
print(otimizadores_df)

print("\n===== LEARNING RATE E BATCH SIZE =====")
print(resultados_df)

##Conclusão

Neste experimento foi desenvolvida uma Rede Neural Feedforward utilizando o dataset MNIST para a classificação de dígitos manuscritos. Inicialmente, os dados foram explorados e preparados, sendo realizada a transformação das imagens de 28 × 28 pixels em vetores de 784 posições, seguida da normalização dos valores dos pixels e da aplicação do One-Hot Encoding nos rótulos.

A rede neural foi construída utilizando duas camadas ocultas, com 256 e 128 neurônios, respectivamente, e uma camada de saída com 10 neurônios. Durante o treinamento foram analisados o comportamento da função de custo e da acurácia nos conjuntos de treinamento e validação.

Também foram realizados experimentos envolvendo diferentes quantidades de épocas, treinamento sem normalização, alterações no learning rate e batch size, funções de ativação e diferentes algoritmos de otimização. Essas comparações permitiram observar como a alteração dos hiperparâmetros influencia o desempenho da rede.

Por fim, o modelo foi avaliado utilizando o conjunto de teste, além da análise por meio da matriz de confusão e das métricas de classificação. Os resultados obtidos demonstram a capacidade da Rede Neural Feedforward em identificar padrões presentes nas imagens do MNIST e realizar a classificação dos dígitos manuscritos.
